# Data Cleaning — Saphondanai (wk1)

**โน้ตบุ๊กนี้ทำอะไร:** เตรียมข้อมูลพนักงานดิบให้ "สะอาด" พอจะเอาไปเทรนโมเดลทำนายการลาออกได้ในสัปดาห์ถัดไป ข้อมูลดิบที่ได้จาก Kaggle ยังเป็นตัวหนังสือ (เช่น "Yes"/"No", "Sales", "Married") ซึ่งโมเดล ML อ่านไม่ได้ ต้องแปลงเป็นตัวเลขก่อน และต้องเช็คก่อนด้วยว่าไม่มีคอลัมน์ไหน "แอบให้คำตอบ" โมเดลล่วงหน้า (data leakage)

**อ่านโน้ตบุ๊กนี้ตามลำดับได้เลย แต่ละหัวข้อจะอธิบายว่า:**
1. กำลังจะทำอะไร
2. ทำไมต้องทำ (เหตุผล)
3. ผลลัพธ์ที่ควรเห็นคืออะไร

**งานนี้เป็นส่วนหนึ่งของ wk1** ตามแผนงานใน [TASKS.md](../TASKS.md) — ผลลัพธ์สุดท้ายของโน้ตบุ๊กนี้จะถูกเทียบกับ `01_cleaning_P.ipynb` ของ Puripat แล้วรวมเป็นไฟล์เดียวที่ [`../src/clean_pipeline.py`](../src/clean_pipeline.py) เพื่อให้ทั้งทีมใช้ต่อได้

> ดูกติกาการเช็ค leakage แบบเต็มได้ที่ README [หัวข้อ 6.2 Data Leakage Guard](../README.md#62-data-leakage-guard)

---
## ขั้นตอนที่ 1 — เตรียมเครื่องมือและหาไฟล์ข้อมูลดิบ

ข้อมูลดิบ (IBM HR Analytics Employee Attrition dataset) อยู่บน Kaggle ไม่ได้เก็บไว้ใน repo ตั้งแต่แรก เพราะฉะนั้นทุกครั้งที่รันโน้ตบุ๊กนี้ครั้งแรกในเครื่องใหม่ โค้ดจะ:

1. เช็คก่อนว่ามีไฟล์อยู่ที่ `data/raw/` แล้วหรือยัง
2. ถ้ายังไม่มี → ดาวน์โหลดจาก Kaggle มาเก็บไว้ให้ (ครั้งเดียวพอ)
3. ถ้ามีแล้ว → ใช้ไฟล์เดิม ไม่โหลดซ้ำให้เสียเวลา

**สิ่งที่ต้องเตรียมก่อนรันได้จริง:** ต้องมีบัญชี Kaggle (ฟรี) และตั้งค่า API token (`kaggle.json`) ไว้ในเครื่องก่อน — วิธีตั้งค่าดูได้ที่ [kagglehub docs](https://github.com/Kaggle/kagglehub) ถ้ายังไม่ตั้ง ตอนรัน cell ถัดไปจะ error เรื่อง authentication

> **หมายเหตุ:** ไฟล์ดิบนี้จะไม่ถูกแก้ไขตรง ๆ เด็ดขาด (read-only) — ทุกการแก้ข้อมูลจะทำใน DataFrame แยกต่างหาก แล้วค่อยเซฟผลลัพธ์ไปไว้ที่ `data/processed/` ตามโครงสร้างที่ตกลงกันไว้ใน README

In [16]:
import os
import shutil
import sys

import pandas as pd

# เพิ่ม root ของโปรเจกต์เข้า path เพื่อ import โมดูล cleaning กลางจาก ../src
sys.path.append(os.path.abspath(".."))
from src.clean_pipeline import (
    check_missing,
    clean_data,
    drop_noise_columns,
    encode_categoricals,
    encode_target,
    save_processed,
)

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
RAW_FILENAME = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
RAW_PATH = os.path.join(RAW_DIR, RAW_FILENAME)

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [17]:
if not os.path.exists(RAW_PATH):
    import kagglehub

    dataset_dir = kagglehub.dataset_download(
        "pavansubhasht/ibm-hr-analytics-attrition-dataset"
    )
    downloaded_csv = os.path.join(dataset_dir, RAW_FILENAME)
    shutil.copy(downloaded_csv, RAW_PATH)
    print(f"ดาวน์โหลด dataset ไปที่ {dataset_dir} แล้วสำเนามาเก็บไว้ที่ {RAW_PATH}")
else:
    print(f"พบไฟล์เดิมอยู่แล้ว ใช้ไฟล์ที่ {RAW_PATH}")

df_raw = pd.read_csv(RAW_PATH)
df_raw.shape  # ควรได้ (1470, 35) -> พนักงาน 1,470 คน 35 คอลัมน์

พบไฟล์เดิมอยู่แล้ว ใช้ไฟล์ที่ ..\data\raw\WA_Fn-UseC_-HR-Employee-Attrition.csv


(1470, 35)

---
## ขั้นตอนที่ 2 — ทำความรู้จักข้อมูลก่อนเริ่มแก้อะไร

ก่อนจะ clean ข้อมูล ต้องดูก่อนว่าหน้าตาข้อมูลเป็นยังไง มีคอลัมน์อะไรบ้าง ชนิดข้อมูลแต่ละคอลัมน์คืออะไร (ตัวเลข/ตัวหนังสือ) และค่าสถิติเบื้องต้นผิดปกติไหม — สามข้อนี้ช่วยจับข้อผิดพลาดได้ตั้งแต่เนิ่น ๆ ก่อนไปเสียเวลากับขั้นตอนถัดไป

In [18]:
# ดูตัวอย่างข้อมูล 5 แถวแรก เพื่อดูหน้าตาคอลัมน์และค่าจริง
df_raw.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [19]:
# ดูชนิดข้อมูล (dtype) ของแต่ละคอลัมน์ — คอลัมน์ไหนเป็นตัวหนังสือ (str/object) คือคอลัมน์ที่ต้องเข้ารหัสในขั้นตอนที่ 6
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

In [20]:
# ดูสถิติเบื้องต้นของทุกคอลัมน์ (ค่าเฉลี่ย, ค่าต่ำสุด-สูงสุด, จำนวนค่าที่ไม่ซ้ำกัน ฯลฯ)
df_raw.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Age,1470.0,NaN,NaN,NaN,36.92381,9.135373,18.0,30.0,36.0,43.0,60.0
Attrition,1470,2,No,1233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BusinessTravel,1470,3,Travel_Rarely,1043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DailyRate,1470.0,NaN,NaN,NaN,802.485714,403.5091,102.0,465.0,802.0,1157.0,1499.0
Department,1470,3,Research & Development,961,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DistanceFromHome,1470.0,NaN,NaN,NaN,9.192517,8.106864,1.0,2.0,7.0,14.0,29.0
Education,1470.0,NaN,NaN,NaN,2.912925,1.024165,1.0,2.0,3.0,4.0,5.0
EducationField,1470,6,Life Sciences,606,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EmployeeCount,1470.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
EmployeeNumber,1470.0,NaN,NaN,NaN,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0


---
## ขั้นตอนที่ 3 — เช็คว่ามีข้อมูลขาดหาย (Missing Value) ไหม

ถ้าบางแถวบางคอลัมน์ไม่มีค่า (เช่น ไม่กรอกเงินเดือน) โมเดลจะเทรนไม่ได้จนกว่าจะเติมค่าหรือลบแถวนั้นทิ้ง จึงต้องเช็คก่อนเสมอ

ไฟล์ CSV ต้นฉบับของ IBM ที่ใช้ในโปรเจกต์นี้ **ไม่มี missing value อยู่แล้ว** แต่เรายังคงเขียนโค้ดเช็คไว้ ไม่ใช่แค่สมมติเอาเอง เพราะถ้าวันหนึ่งมีใครเปลี่ยนไปใช้ข้อมูลชุดอื่น (เช่น ข้อมูลจริงของบริษัทตอน recalibrate) โค้ดจุดนี้จะจับปัญหาได้ทันที

In [21]:
missing = check_missing(df_raw)
missing[missing > 0]  # ผลลัพธ์ควรเป็นตารางว่าง = ไม่มีคอลัมน์ไหนมี missing value เลย

Series([], dtype: int64)

---
## ขั้นตอนที่ 4 — เช็ค Data Leakage และตัดคอลัมน์ที่ไม่มีประโยชน์

**Data Leakage คืออะไร (แบบเข้าใจง่าย):** ถ้าเอาข้อมูลที่ "รู้ได้หลังจากพนักงานลาออกไปแล้ว" มาเทรนโมเดล โมเดลจะดูแม่นมากตอนทดสอบ แต่พอเอาไปใช้จริงกับพนักงานที่ยังทำงานอยู่ (ซึ่งเรายังไม่รู้อนาคต) โมเดลจะทำนายผิดเพราะไม่เคยเห็นข้อมูลแบบนี้จริง ๆ มาก่อน — กติกาคือ **ต้องใช้แค่ข้อมูลที่รู้ ณ ตอนที่พนักงานยังทำงานอยู่เท่านั้น**

ไล่ตรวจครบทั้ง 35 คอลัมน์แล้ว สรุปว่า:

- **ไม่มี leakage เชิงเวลา** — dataset ชุดนี้เป็นข้อมูล ณ จุดเวลาเดียว (snapshot) ของพนักงานแต่ละคน ไม่มีคอลัมน์ไหนที่ถูกเติมค่าเฉพาะ *หลัง* เหตุการณ์ลาออกเกิดขึ้น (ไม่มีข้อความจาก exit interview, ไม่มีฟีเจอร์คำนวณจากวันที่ออกงาน)
- แต่พบ **4 คอลัมน์ที่ไม่มีประโยชน์** (ไม่ใช่ leakage แต่เป็น "noise") ต้องตัดทิ้งก่อนเทรนโมเดล:

| คอลัมน์ | ทำไมถึงตัดทิ้ง |
| :--- | :--- |
| `EmployeeCount` | ค่าเดียวกันหมดทุกแถว (=1) — ไม่ช่วยแยกแยะอะไรเลย |
| `StandardHours` | ค่าเดียวกันหมดทุกแถว (=80) |
| `Over18` | ค่าเดียวกันหมดทุกแถว (='Y') |
| `EmployeeNumber` | เป็นแค่เลขรันประจำตัวพนักงาน ไม่ใช่ปัจจัยที่บอกอะไรเกี่ยวกับพฤติกรรมการลาออก |

In [22]:
# ยืนยันด้วยสายตาว่า 3 คอลัมน์นี้มีค่าเดียวจริง ๆ (unique value มีแค่ 1 ค่า)
for col in ["EmployeeCount", "StandardHours", "Over18"]:
    print(col, df_raw[col].unique())

EmployeeCount [1]
StandardHours [80]
Over18 <StringArray>
['Y']
Length: 1, dtype: str


In [23]:
df = drop_noise_columns(df_raw)
df.shape  # จำนวนคอลัมน์ควรลดลงจาก 35 เหลือ 31 (ตัดไป 4 คอลัมน์)

(1470, 31)

---
## ขั้นตอนที่ 5 — แปลงคำตอบ (`Attrition`) ให้เป็นตัวเลข

`Attrition` คือคอลัมน์เป้าหมายที่โมเดลต้องทำนาย ตอนนี้เก็บเป็นตัวหนังสือ "Yes"/"No" ต้องแปลงเป็น 1/0 ก่อน เพราะโมเดล ML รับได้แค่ตัวเลข (1 = ลาออก, 0 = ไม่ลาออก)

In [24]:
df = encode_target(df)
df["Attrition"].value_counts(normalize=True)  # ควรเห็นสัดส่วนราว 84% (0) : 16% (1) ตรงกับที่ README ระบุไว้

Attrition
0    0.838776
1    0.161224
Name: proportion, dtype: float64

---
## ขั้นตอนที่ 6 — แปลงคอลัมน์หมวดหมู่ (ตัวหนังสือ) ให้เป็นตัวเลข

เลือกวิธีแปลงตามลักษณะของแต่ละคอลัมน์ ไม่ใช้วิธีเดียวกันหมด เพราะบางคอลัมน์ "มีลำดับ" บางคอลัมน์ "ไม่มีลำดับ":

| ลักษณะคอลัมน์ | คอลัมน์ | วิธีแปลง | เหตุผล |
| :--- | :--- | :--- | :--- |
| มีลำดับ (ordinal) | `BusinessTravel` | Non-Travel=0, Travel_Rarely=1, Travel_Frequently=2 | รักษาลำดับความถี่การเดินทางไว้ ให้โมเดลรู้ว่า 2 มากกว่า 1 จริง ๆ |
| มี 2 ค่า (binary) | `OverTime`, `Gender` | แปลงตรงเป็น 0/1 | มีแค่ 2 ตัวเลือก ไม่ต้องทำอะไรซับซ้อน |
| ไม่มีลำดับ (nominal) | `Department`, `EducationField`, `JobRole`, `MaritalStatus` | one-hot encoding (แตกเป็นหลายคอลัมน์ 0/1) | ถ้าใช้ตัวเลขเรียงกันเฉย ๆ โมเดลจะเข้าใจผิดว่ามีลำดับความสำคัญ ทั้งที่จริงไม่มี |

ส่วนฟีเจอร์แบบ Likert scale เช่น `Education`, `JobSatisfaction`, `WorkLifeBalance` ฯลฯ เป็นตัวเลข 1-4 อยู่แล้วในไฟล์ต้นฉบับ ไม่ต้องแปลงเพิ่ม

In [25]:
df_clean = encode_categoricals(df)
df_clean.shape  # จำนวนคอลัมน์จะเพิ่มขึ้นจากตอนตัด noise เพราะ one-hot แตกคอลัมน์ nominal ออกเป็นหลายคอลัมน์

(1470, 48)

In [26]:
# ดูตัวอย่างข้อมูลหลัง clean ครบทุกขั้นตอน — ทุกคอลัมน์ควรเป็นตัวเลขล้วนแล้ว ไม่มีตัวหนังสือเหลืออยู่
df_clean.head()

,Age,Attrition,BusinessTravel,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,...,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single
0,41,1,1,1102,1,2,2,0,94,3,...,0,0,0,0,0,1,0,0,0,1
1,49,0,2,279,8,1,3,1,61,2,...,0,0,0,0,1,0,0,0,1,0
2,37,1,1,1373,2,2,4,1,92,2,...,1,0,0,0,0,0,0,0,0,1
3,33,0,2,1392,3,4,4,0,56,3,...,0,0,0,0,1,0,0,0,1,0
4,27,0,1,591,2,1,1,1,40,3,...,1,0,0,0,0,0,0,0,1,0


---
## ขั้นตอนที่ 7 — เช็คว่าโค้ดในโน้ตบุ๊กตรงกับโมดูลกลางจริง ๆ

ทุกขั้นตอนด้านบน (ตัดคอลัมน์ noise, เข้ารหัส target, เข้ารหัส categorical) ถูกเขียนซ้ำอีกครั้งเป็นฟังก์ชันเดียวชื่อ `clean_data()` ไว้ที่ [`../src/clean_pipeline.py`](../src/clean_pipeline.py) เพื่อให้ทีม feature engineering ใน wk2-3 เรียกใช้ต่อได้เลยโดยไม่ต้องกลับมาอ่านโน้ตบุ๊กนี้ทั้งไฟล์

เพราะฉะนั้นก่อนเซฟผลลัพธ์ ต้องมั่นใจว่าฟังก์ชันกลางให้ผลลัพธ์ตรงกับที่ทำทีละขั้นตอนในโน้ตบุ๊กนี้ทุกประการ — ถ้า assert ผ่านแบบไม่ error แปลว่าตรงกัน

In [27]:
df_clean_via_pipeline = clean_data(df_raw)
assert df_clean.equals(df_clean_via_pipeline), "ผลลัพธ์จากขั้นตอนด้วยมือกับ clean_data() ไม่ตรงกัน"
print("OK: ผลลัพธ์จาก clean_pipeline.clean_data() ตรงกับขั้นตอนด้านบนทุกประการ")

OK: ผลลัพธ์จาก clean_pipeline.clean_data() ตรงกับขั้นตอนด้านบนทุกประการ


---
## ขั้นตอนที่ 8 — บันทึกผลลัพธ์

เซฟข้อมูลที่ clean เสร็จแล้วเป็นไฟล์ CSV ใหม่ใน `data/processed/` (ไม่แตะไฟล์ดิบใน `data/raw/` เลย) ไฟล์นี้แหละที่จะถูกใช้ต่อในขั้นตอน feature engineering และเทรนโมเดล

In [28]:
processed_path = os.path.join(PROCESSED_DIR, "attrition_cleaned_S.csv")
save_processed(df_clean, processed_path)
processed_path

'..\\data\\processed\\attrition_cleaned_S.csv'

---
## สรุปสิ่งที่ทำในโน้ตบุ๊กนี้

| หัวข้อ | ผลลัพธ์ |
| :--- | :--- |
| Missing value | ไม่พบเลย ไม่ต้องเติม/ลบข้อมูล |
| Data leakage | ไม่พบ (dataset เป็น snapshot ณ จุดเวลาเดียว) |
| คอลัมน์ noise ที่ตัดทิ้ง | `EmployeeCount`, `StandardHours`, `Over18`, `EmployeeNumber` (4 คอลัมน์) |
| การเข้ารหัส | `Attrition` → 0/1, `BusinessTravel` → ordinal, `OverTime`/`Gender` → binary, `Department`/`EducationField`/`JobRole`/`MaritalStatus` → one-hot |
| ไฟล์ผลลัพธ์ | `data/processed/attrition_cleaned_S.csv` |

**ขั้นตอนถัดไป (ยังไม่เสร็จในโน้ตบุ๊กนี้):**
1. เทียบผลลัพธ์แบบคอลัมน์ต่อคอลัมน์กับ `01_cleaning_P.ipynb` ของ Puripat
2. ตกลงกันว่าจะใช้วิธีไหนถ้าผลลัพธ์ไม่ตรงกัน
3. ยืนยันให้ `src/clean_pipeline.py` เป็นไฟล์ cleaning กลางไฟล์เดียวที่ทั้งทีมใช้ต่อในงาน feature engineering ของ wk2-3